# 🌱 Soil Recommendation — Exploratory Data Analysis (EDA)
### Comprehensive data analysis before Fine-tuning
---
| | |
|---|---|
| Dataset | merged_dataset_balanced.csv |
| Rows | 61,791 |
| Crops | 78 crops |
| Sources | 7 sources (Global + MALR Egypt) |

## 📦 STEP 1 — Install Libraries

In [ ]:
!pip install matplotlib seaborn plotly -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# unified style
plt.rcParams['figure.dpi']      = 120
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor']   = '#f8f9fa'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.alpha']       = 0.4
plt.rcParams['font.size']        = 11
PALETTE = 'viridis'
GREEN   = '#1D9E75'
BLUE    = '#378ADD'
ORANGE  = '#EF9F27'
RED     = '#E24B4A'

print('✅ Libraries ready')

## 📂 STEP 2 — Load Data

In [ ]:
from google.colab import files
print('Upload merged_dataset_balanced.csv:')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(fname)

NUM_COLS  = ['nitrogen','phosphorus','potassium','ph','temperature','humidity','rainfall','moisture']
CAT_COLS  = ['crop','fertilizer','soil_type','source']

print(f'✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print(f'Crops    : {df["crop"].nunique()}')
print(f'Sources  : {df["source"].nunique()}')
df.head()

## 📊 STEP 3 — Dataset Overview
### Basic Statistics + NULL Check

In [ ]:
print('=== Basic Statistics ===')
display(df[NUM_COLS].describe().round(2))

print('\n=== NULL Check ===')
nulls = df.isnull().sum()
if nulls.sum() == 0:
    print('✅ No NULL values in dataset')
else:
    print(nulls[nulls > 0])

print('\n=== Data Types ===')
print(df.dtypes)

print('\n=== Source Distribution ===')
print(df['source'].value_counts())

## 📈 STEP 4 — A) Feature Distributions (Histograms)
### Distribution of N, P, K, pH and other features

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

colors = [GREEN, BLUE, ORANGE, RED, '#7F77DD', '#D4537E', '#639922', '#BA7517']
labels = [
    'Nitrogen (N) — kg/ha',
    'Phosphorus (P) — kg/ha',
    'Potassium (K) — kg/ha',
    'Soil pH',
    'Temperature — °C',
    'Relative Humidity — %',
    'Rainfall — mm',
    'Soil Moisture — %'
]

for i, (col, label, color) in enumerate(zip(NUM_COLS, labels, colors)):
    ax = axes[i]
    data = df[col].dropna()
    ax.hist(data, bins=50, color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
    ax.axvline(data.mean(),   color='red',    linestyle='--', linewidth=1.5, label=f'Mean: {data.mean():.1f}')
    ax.axvline(data.median(), color='black',  linestyle=':',  linewidth=1.5, label=f'Median: {data.median():.1f}')
    ax.set_title(label, fontsize=10, fontweight='bold', pad=8)
    ax.legend(fontsize=8)
    ax.set_ylabel('Frequency')

plt.suptitle('A) Numerical Feature Distributions', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('EDA_A_distributions.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_A_distributions.png')

## 🔥 STEP 5 — B) Correlation Matrix
### Correlation between numerical columns

In [ ]:
corr = df[NUM_COLS].corr().round(2)

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)

sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdYlGn',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 9}
)

labels_ar = ['Nitrogen','Phosphorus','Potassium','pH','Temperature','Humidity','Rainfall','Soil Moisture']
ax.set_xticklabels(labels_ar, rotation=45, ha='right', fontsize=9)
ax.set_yticklabels(labels_ar, rotation=0, fontsize=9)
ax.set_title('B) Correlation Matrix — Relationships Between Features', fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('EDA_B_correlation.png', bbox_inches='tight', dpi=150)
plt.show()

# strongest correlations
print('\nStrongest correlations (|r| > 0.3):')
corr_pairs = corr.unstack().reset_index()
corr_pairs.columns = ['Feature1','Feature2','Correlation']
corr_pairs = corr_pairs[
    (corr_pairs['Feature1'] != corr_pairs['Feature2']) &
    (corr_pairs['Feature1'] < corr_pairs['Feature2']) &
    (corr_pairs['Correlation'].abs() > 0.3)
].sort_values('Correlation', key=abs, ascending=False)
print(corr_pairs.to_string(index=False))
print('\n✅ Saved: EDA_B_correlation.png')

## 📦 STEP 6 — C) Box Plots
### N, P, K, pH distribution for top crops

In [ ]:
# Get top 20 crops (most common)
top_crops = df['crop'].value_counts().head(20).index.tolist()
df_top    = df[df['crop'].isin(top_crops)]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))

box_cols   = ['nitrogen', 'phosphorus', 'potassium', 'ph']
box_labels = ['Nitrogen (N) — kg/ha', 'Phosphorus (P) — kg/ha',
              'Potassium (K) — kg/ha', 'Soil pH']
box_colors = [GREEN, BLUE, ORANGE, RED]

for ax, col, label, color in zip(axes.flatten(), box_cols, box_labels, box_colors):
    order = df_top.groupby('crop')[col].median().sort_values(ascending=False).index
    sns.boxplot(
        data=df_top, x='crop', y=col,
        order=order, color=color, ax=ax,
        width=0.6, linewidth=0.8,
        flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax.set_title(f'C) {label}', fontsize=11, fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(label.split('—')[0].strip())
    ax.tick_params(axis='x', rotation=45, labelsize=8)

plt.suptitle('C) Box Plots — Distribution elements for top 20 crops', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('EDA_C_boxplots.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_C_boxplots.png')

## 📊 STEP 7 — D) Class Distribution
### Crop distribution before and after Balancing

In [ ]:
dist = df['crop'].value_counts().reset_index()
dist.columns = ['crop','count']
dist['color'] = dist['count'].apply(
    lambda x: RED if x < 400 else (ORANGE if x < 800 else GREEN)
)

fig, ax = plt.subplots(figsize=(18, 10))
bars = ax.barh(dist['crop'], dist['count'],
               color=dist['color'], edgecolor='white', linewidth=0.3, height=0.7)

for bar, count in zip(bars, dist['count']):
    ax.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
            f'{count:,}', va='center', ha='left', fontsize=7)

ax.axvline(300,  color=RED,    linestyle='--', linewidth=1, alpha=0.7, label='Minimum (300)')
ax.axvline(dist['count'].mean(), color=BLUE, linestyle=':', linewidth=1.5,
           label=f'Mean ({int(dist["count"].mean()):,})')

ax.set_xlabel('Row Count', fontsize=11)
ax.set_title('D) Class Distribution — Crop Distribution in Dataset', fontsize=13, fontweight='bold', pad=15)
ax.legend(fontsize=10)
ax.invert_yaxis()

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=GREEN,  label='Balanced (800+)'),
    Patch(facecolor=ORANGE, label='Acceptable (400-800)'),
    Patch(facecolor=RED,    label='Low (< 400)'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('EDA_D_class_distribution.png', bbox_inches='tight', dpi=150)
plt.show()

print(f'least crops  : {dist.iloc[-1]["crop"]} ({dist.iloc[-1]["count"]} rows)')
print(f'most crops : {dist.iloc[0]["crop"]}  ({dist.iloc[0]["count"]} rows)')
print(f'Ratio     : {dist.iloc[0]["count"]//dist.iloc[-1]["count"]}x')
print('\n✅ Saved: EDA_D_class_distribution.png')

## 🎯 STEP 8 — E) Outlier Detection
### Outlier detection per column

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()

outlier_summary = []

for i, (col, label, color) in enumerate(zip(NUM_COLS, labels, colors)):
    ax   = axes[i]
    data = df[col].dropna()
    Q1   = data.quantile(0.25)
    Q3   = data.quantile(0.75)
    IQR  = Q3 - Q1
    lo   = Q1 - 1.5 * IQR
    hi   = Q3 + 1.5 * IQR
    n_out = ((data < lo) | (data > hi)).sum()
    pct   = n_out / len(data) * 100
    outlier_summary.append({'Feature': col, 'Outliers': n_out, '%': round(pct,2)})

    bp = ax.boxplot(data, vert=True, patch_artist=True,
                    boxprops=dict(facecolor=color, alpha=0.7),
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='o', color=RED, markersize=3, alpha=0.4))
    ax.set_title(f'{label}\nOutliers: {n_out:,} ({pct:.1f}%)',
                 fontsize=9, fontweight='bold')
    ax.set_xticks([])

plt.suptitle('E) Outlier Detection — Outliers per Column', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('EDA_E_outliers.png', bbox_inches='tight', dpi=150)
plt.show()

print('\n=== Outlier Summary ===')
out_df = pd.DataFrame(outlier_summary)
print(out_df.to_string(index=False))
print('\n✅ Saved: EDA_E_outliers.png')

## 🔗 STEP 9 — F) Feature Importance
### Feature importance for crop classification

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

le  = LabelEncoder()
X   = df[NUM_COLS].fillna(df[NUM_COLS].median())
y   = le.fit_transform(df['crop'])

rf  = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)

imp = pd.DataFrame({
    'Feature':    NUM_COLS,
    'Label':      labels,
    'Importance': rf.feature_importances_
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
bar_colors = [GREEN if v > 0.15 else BLUE if v > 0.08 else ORANGE
              for v in imp['Importance']]
bars = ax.barh(imp['Label'], imp['Importance'],
               color=bar_colors, edgecolor='white', height=0.6)

for bar, val in zip(bars, imp['Importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)

ax.set_xlabel('Feature Importance (Random Forest)', fontsize=11)
ax.set_title('F) Feature Importance — Impact on Crop Classification',
             fontsize=12, fontweight='bold', pad=15)
ax.set_xlim(0, imp['Importance'].max() * 1.15)

plt.tight_layout()
plt.savefig('EDA_F_feature_importance.png', bbox_inches='tight', dpi=150)
plt.show()

print('\nMost important features for crop classification:')
print(imp[['Label','Importance']].sort_values('Importance', ascending=False).to_string(index=False))
print('\n✅ Saved: EDA_F_feature_importance.png')

## 🌍 STEP 10 — G) Source Distribution
### Egyptian vs Global data ratio

In [ ]:
source_dist = df['source'].value_counts().reset_index()
source_dist.columns = ['source','count']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
palette = [GREEN, BLUE, ORANGE, RED, '#7F77DD', '#D4537E', '#639922']
wedges, texts, autotexts = ax1.pie(
    source_dist['count'],
    labels=source_dist['source'],
    autopct='%1.1f%%',
    colors=palette[:len(source_dist)],
    startangle=90,
    pctdistance=0.75,
    textprops={'fontsize': 9}
)
for at in autotexts: at.set_fontsize(8)
ax1.set_title('G) Source Distribution — Pie Chart', fontweight='bold')

# Bar chart
bars = ax2.barh(source_dist['source'], source_dist['count'],
                color=palette[:len(source_dist)], edgecolor='white', height=0.6)
for bar, count in zip(bars, source_dist['count']):
    ax2.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
             f'{count:,}', va='center', fontsize=9)
ax2.set_xlabel('Row Count')
ax2.set_title('G) Source Distribution — Row Count', fontweight='bold')
ax2.invert_yaxis()

plt.suptitle('G) Data Sources', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('EDA_G_sources.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_G_sources.png')

## 🧪 STEP 11 — H) Distribution Fertilizers
### Most common fertilizers in dataset

In [ ]:
fert_dist = df['fertilizer'].value_counts().head(15).reset_index()
fert_dist.columns = ['fertilizer','count']

fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.bar(range(len(fert_dist)), fert_dist['count'],
              color=[GREEN if i < 3 else BLUE if i < 7 else ORANGE
                     for i in range(len(fert_dist))],
              edgecolor='white', linewidth=0.5)

for bar, count in zip(bars, fert_dist['count']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(range(len(fert_dist)))
ax.set_xticklabels(fert_dist['fertilizer'], rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Row Count')
ax.set_title('H) Fertilizer Distribution — Top 15 Fertilizers', fontsize=12, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('EDA_H_fertilizers.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_H_fertilizers.png')

## 🌱 STEP 12 — I) NPK Profile for Top Crops
### Comparison average N, P, K each crops

In [ ]:
top15 = df['crop'].value_counts().head(15).index.tolist()
npk   = df[df['crop'].isin(top15)].groupby('crop')[['nitrogen','phosphorus','potassium']].mean().round(1)
npk   = npk.sort_values('nitrogen', ascending=False)

x     = np.arange(len(npk))
w     = 0.25

fig, ax = plt.subplots(figsize=(16, 7))
b1 = ax.bar(x - w,   npk['nitrogen'],   w, label='Nitrogen (N)', color=GREEN,  alpha=0.85)
b2 = ax.bar(x,       npk['phosphorus'], w, label='Phosphorus (P)',   color=BLUE,   alpha=0.85)
b3 = ax.bar(x + w,   npk['potassium'],  w, label='Potassium (K)', color=ORANGE, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(npk.index, rotation=40, ha='right', fontsize=9)
ax.set_ylabel('Average (kg/ha)')
ax.set_title('I) NPK Profile — average elements Soil top 15 crops',
             fontsize=12, fontweight='bold', pad=15)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('EDA_I_npk_profile.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_I_npk_profile.png')

## 🌡️ STEP 13 — J) pH Distribution each crops
### Violin Plot to understand range tolerance each crops for acidity

In [ ]:
top20   = df['crop'].value_counts().head(20).index.tolist()
df_top20 = df[df['crop'].isin(top20)]
order_ph = df_top20.groupby('crop')['ph'].median().sort_values().index

fig, ax = plt.subplots(figsize=(18, 7))
sns.violinplot(
    data=df_top20, x='crop', y='ph',
    order=order_ph, palette='viridis',
    inner='quartile', ax=ax, linewidth=0.8
)
ax.axhline(7.0, color=RED,    linestyle='--', linewidth=1.5, label='Neutral pH (7.0)')
ax.axhline(6.5, color=ORANGE, linestyle=':',  linewidth=1.5, label='Ideal pH for most crops (6.5)')
ax.set_xlabel('')
ax.set_ylabel('Soil pH')
ax.set_title('J) Distribution Soil pH for top 20 crops', fontsize=12, fontweight='bold', pad=15)
ax.tick_params(axis='x', rotation=45, labelsize=8)
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('EDA_J_ph_violin.png', bbox_inches='tight', dpi=150)
plt.show()
print('✅ Saved: EDA_J_ph_violin.png')

## 📋 STEP 14 — K) Full EDA Summary
### Key insights for discussion

In [ ]:
print('=' * 60)
print('         EDA Summary — Soil Recommendation Dataset')
print('=' * 60)

print(f'\n📊 Dataset Size:')
print(f'   • {len(df):,} rows | {df["crop"].nunique()} crops | {df["source"].nunique()} sources')
print(f'   • 0 NULL values in any column')

print(f'\n🌱 Top Features (Feature Importance):')
print(f'   • Nitrogen and Phosphorus are the most discriminative features')
print(f'   • pH is a key factor in determining suitable soil type')

print(f'\n⚖️ Class Balance:')
dist = df['crop'].value_counts()
print(f'   • most crops: {dist.idxmax()} ({dist.max():,} rows)')
print(f'   • least crops: {dist.idxmin()} ({dist.min():,} rows)')
print(f'   • Imbalance ratio: {dist.max()//dist.min()}x (after Balancing)')

print(f'\n🇪🇬 Egyptian Data:')
egypt_rows = len(df[df['source'] == 'MALR-Egypt'])
print(f'   • {egypt_rows:,} rows from MALR Egypt ({egypt_rows/len(df)*100:.1f}% of total)')
print(f'   • {df[df["source"]=="MALR-Egypt"]["crop"].nunique()} crops Egyptian native')

print(f'\n🧪 Most Common Fertilizers:')
print(df['fertilizer'].value_counts().head(5).to_string())

print(f'\n🏗️ Most Common Soil Types:')
print(df['soil_type'].value_counts().head(5).to_string())

print(f'\n✅ Data is ready for Fine-tuning on Mistral-7B + QLoRA!')
print('=' * 60)

## 📤 STEP 15 — Save All Plots

In [ ]:
import zipfile, os

png_files = [f for f in os.listdir('.') if f.startswith('EDA_') and f.endswith('.png')]

with zipfile.ZipFile('EDA_plots.zip', 'w') as zf:
    for fn in sorted(png_files):
        zf.write(fn)
        print(f'  ✅ {fn}')

files.download('EDA_plots.zip')
print(f'\n✅ {len(png_files)} plots saved in EDA_plots.zip')